In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore')

font_path      = 'font-folder/Roboto-Light.ttf'
font_path_bold = 'font-folder/Roboto-Bold.ttf'
font_prop         = fm.FontProperties(fname=font_path,      size=12)
font_prop_small   = fm.FontProperties(fname=font_path,      size=10)
font_prop_numbers = fm.FontProperties(fname=font_path,      size=9)
font_prop_title   = fm.FontProperties(fname=font_path_bold, size=14)

In [ ]:
# ============================================================
# SETTINGS
# ============================================================
challenge          = 'FluSight'
baseline_challenge = 'FluSight-baseline'
metric             = 'wis_relative'
metric_label       = 'WIS'

TARGET_FILE = '../target-data/FluSight/surveillance_weekly.csv'

if challenge == 'FluSight':
    SCORES_FILE = f'../model-evaluation/forecast_scores/forecast_scores_wis_ae_coverage_log_{challenge}.csv'
else:
    SCORES_FILE = f'../model-evaluation/forecast_scores/forecast_scores_wis_ae_coverage_log_{challenge}.csv'

tsfm_models = [
    'Chronos-2', 'Lag-Llama',
    'Moirai-2',  'TabPFN-TS',
    'TimeGPT', 'TimesFM'
]

model_palettes = {
    'Chronos-2':         '#4ECED4',
    'Moirai-2':          '#967ECD',
    'Lag-Llama':         '#3E977E',
    'TabPFN-TS':         '#CE5858',
    'TimeGPT':           '#1F77B4',
    'TimesFM':           '#F2B134',
    'FluSight-ensemble': 'teal',
}

In [ ]:
# ============================================================
# LOAD SCORES — all locations
# ============================================================
df_scores = pd.read_csv(SCORES_FILE)
df_scores['reference_date'] = pd.to_datetime(df_scores['reference_date'])

# relative scores vs baseline
baseline_df = df_scores[df_scores['model'] == baseline_challenge][
    ['reference_date', 'target', 'target_end_date', 'horizon', 'location', 'wis', 'ae_median']
].rename(columns={'wis': 'wis_baseline', 'ae_median': 'ae_median_baseline'})

df_merged = df_scores.merge(
    baseline_df,
    on=['reference_date', 'target', 'target_end_date', 'horizon', 'location'],
    how='left'
)
df_merged['wis_relative']       = df_merged['wis']       / df_merged['wis_baseline']
df_merged['ae_median_relative'] = df_merged['ae_median'] / df_merged['ae_median_baseline']
df_merged = df_merged.drop(columns=['wis_baseline', 'ae_median_baseline'])

df_merged['model_name'] = df_merged['model'].copy()

print(f'Scores loaded: {len(df_merged)} rows')
print(f'Locations: {df_merged["location"].nunique()}')
print(f'Models: {df_merged["model_name"].nunique()}')

In [ ]:
# ============================================================
# LOAD SURVEILLANCE — all locations
# ============================================================
df_surv = pd.read_csv(TARGET_FILE)
df_surv['truth_date'] = pd.to_datetime(df_surv['truth_date'])
df_surv = (
    df_surv
    .drop_duplicates(['truth_date', 'location'])
    .sort_values(['location', 'truth_date'])
    .rename(columns={'value': 'obs'})
    [['truth_date', 'location', 'obs']]
    .copy()
)

In [ ]:
# ============================================================
# LOAD POPULATION (needed to convert obs -> rate per 100k)
# ============================================================
url_locations = "https://raw.githubusercontent.com/cdcepi/FluSight-forecast-hub/main/auxiliary-data/locations.csv"
df_pop = pd.read_csv(url_locations)
print(df_pop.head())
print(df_pop.columns.tolist())
# expect columns like: abbreviation, location, location_name, population

df_pop = df_pop.rename(columns={'location': 'location'})[['location', 'population']]

df_surv = df_surv.merge(df_pop, on='location', how='left')
print("Missing population for:", df_surv[df_surv['population'].isna()]['location'].unique())

# ============================================================
# COMPUTE RAW COUNT DIFF AND RATE DIFF PER LOCATION
# ============================================================
def compute_diff_and_rate(grp):
    grp = grp.sort_values('truth_date').copy()
    grp['obs_lag'] = grp['obs'].shift(1)
    grp['diff'] = grp['obs'] - grp['obs_lag']
    grp['rate'] = grp['obs'] / grp['population'] * 1e5
    grp['rate_lag'] = grp['rate'].shift(1)
    grp['rate_diff'] = grp['rate'] - grp['rate_lag']
    return grp

df_surv = df_surv.groupby('location', group_keys=False).apply(compute_diff_and_rate)
df_surv = df_surv.dropna(subset=['diff', 'rate_diff'])

# ============================================================
# ASSIGN SEASON (unchanged)
# ============================================================
def assign_season(date):
    if date.month >= 10:
        return f"{date.year}/{date.year+1}"
    else:
        return f"{date.year-1}/{date.year}"

df_surv['season'] = df_surv['truth_date'].apply(assign_season)

cat_labels_ordered = ['large_decrease', 'decrease', 'stable', 'increase', 'large_increase']

# ============================================================
# CDC FIXED-THRESHOLD CLASSIFIER (given thresholds, applies to any hub)
# ============================================================
def classify_trend_fixed(df_surv, rate_thresholds, target_seasons=None, count_floor=10):
    """
    rate_thresholds: dict with keys 'large_decrease','decrease','increase','large_increase'
                      (rate/100k cutoffs, e.g. {'large_decrease': -1.7, 'decrease': -0.3,
                                                 'increase': 0.3, 'large_increase': 1.7})
    """
    df_surv = df_surv.copy()
    mask = df_surv['season'].isin(target_seasons) if target_seasons is not None else pd.Series(True, index=df_surv.index)

    rate_diff = df_surv.loc[mask, 'rate_diff']
    count_diff = df_surv.loc[mask, 'diff']

    cat = pd.Series(index=rate_diff.index, dtype=object)
    cat[rate_diff <= rate_thresholds['large_decrease']] = 'large_decrease'
    cat[(rate_diff > rate_thresholds['large_decrease']) & (rate_diff < rate_thresholds['decrease'])] = 'decrease'
    cat[(rate_diff >= rate_thresholds['decrease']) & (rate_diff <= rate_thresholds['increase'])] = 'stable'
    cat[(rate_diff > rate_thresholds['increase']) & (rate_diff < rate_thresholds['large_increase'])] = 'increase'
    cat[rate_diff >= rate_thresholds['large_increase']] = 'large_increase'

    cat[count_diff.abs() < count_floor] = 'stable'

    df_surv['trend_cat'] = np.nan
    df_surv.loc[mask, 'trend_cat'] = cat
    df_surv['trend_cat'] = pd.Categorical(df_surv['trend_cat'], categories=cat_labels_ordered, ordered=True)
    return df_surv


# ============================================================
# CASE 1: FluSight — use official CDC published thresholds directly
# ============================================================
CDC_FLU_THRESHOLDS = {
    'large_decrease': -1.7, 'decrease': -0.3,
    'increase': 0.3, 'large_increase': 1.7
}

TARGET_SEASONS = ['2024/2025', '2025/2026']

df_surv_flu = classify_trend_fixed(
    df_surv[df_surv['challenge'] == 'FluSight'] if 'challenge' in df_surv.columns else df_surv,
    CDC_FLU_THRESHOLDS, target_seasons=TARGET_SEASONS
)

In [ ]:
df_surv_flu = df_surv_flu[df_surv_flu['season'].isin(TARGET_SEASONS)].copy()
df_surv_flu

In [ ]:
df_surv_flu.to_csv('../model-evaluation/df_surv_flu_trend_cat_CDC_official.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.patches as mpatches
import matplotlib.dates as mdates

# scegli location
loc = 'US'
df_plot = df_surv_flu[df_surv_flu['location'] == loc].copy()
if challenge == 'FluSight':
    df_plot = df_plot[df_plot['season'] >= '2024/2025']

cat_colors = {
    'large_decrease': '#2166AC',  # blu scuro — declino ripido
    'decrease':        '#92C5DE',  # azzurro — declino lieve
    'stable':          '#B0B0B0',  # grigio — stabile
    'increase':        '#F4A582',  # arancio — crescita lieve
    'large_increase':  '#D6604D',  # rosso — crescita ripida
}

cat_labels_display = {
    'large_decrease': 'Large decrease',
    'decrease':        'Decrease',
    'stable':          'Stable',
    'increase':        'Increase',
    'large_increase':  'Large increase',
}

fig, ax = plt.subplots(figsize=(12, 4), dpi=200)

# linea di sfondo
ax.plot(df_plot['truth_date'], df_plot['obs'],
        color='#cccccc', linewidth=1.0, zorder=1)

# punti colorati per categoria di trend
for cat, color in cat_colors.items():
    sub = df_plot[df_plot['trend_cat'] == cat]
    ax.scatter(sub['truth_date'], sub['obs'],
               color=color, s=40, zorder=2, label=cat_labels_display[cat])

ax.set_ylabel('Weekly hospital admissions', fontproperties=font_prop)
ax.set_xlabel('Date', fontproperties=font_prop)
ax.set_title(f'{loc} — epidemic trend by category', fontproperties=font_prop_title,
             loc='left', x=0.01)

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.spines['left'].set_linewidth(0.5)
ax.spines['bottom'].set_linewidth(0.5)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right',
         fontproperties=font_prop_numbers)
for lbl in ax.get_yticklabels():
    lbl.set_fontproperties(font_prop_numbers)

legend_handles = [
    mpatches.Patch(facecolor=cat_colors[cat], label=cat_labels_display[cat])
    for cat in cat_labels_ordered
]
ax.legend(handles=legend_handles, frameon=False, prop=font_prop_small,
          loc='upper left')

plt.tight_layout()
plt.show()